# Akan-BPE Model Ladder - Heavy Split

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/run-full-heavy.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/run-full-heavy.ipynb)

Runs the heavier/riskier half of the ladder: Qwen3-1.7B and Gemma-3-1B.

Run this top-to-bottom on Kaggle or Colab with a T4 GPU. Each model runs two arms: random embedding init and mean-of-subword embedding init. After each model, the notebook prints fertility/tokenization metrics, BPB metrics, generation samples, reload verification, the full JSON payloads, and then zips artifacts into `outputs/`.

## Setup

In [1]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

Cloning into 'akan-bpe'...
remote: Enumerating objects: 683, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 683 (delta 106), reused 114 (delta 67), pack-reused 516 (from 1)
Receiving objects: 100% (683/683), 1.22 MiB | 16.68 MiB/s, done.
Resolving deltas: 100% (430/430), done.
/kaggle/working/akan-bpe
Working directory: /kaggle/working/akan-bpe


In [2]:
!pip install -q uv
!uv pip install -q --system -e ".[dev,train]"
!uv pip install -q --system bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 73.8 MB/s eta 0:00:00:00:0100:01


In [3]:
# Hugging Face authentication. Paste a token with access to gated models
# when prompted. The input is hidden.
from huggingface_hub import login

login()

In [4]:
# Confirm a GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a T4 GPU accelerator, then re-run.")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [5]:
# Download Akan datasets. --tts-limit 50000 pins the 40,000/5,000/5,000 split.
!python scripts/download.py --output-dir data --tts-limit 50000

README.md: 29.2kB [00:00, 13.9MB/s]
Resolving data files: 100%|████████████████| 270/270 [00:00<00:00, 32870.72it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 4.06kB [00:00, 11.0MB/s]
Wrote 40000 rows to data/pristine_twi_train.jsonl
Wrote 5000 rows to data/pristine_twi_validation.jsonl
Wrote 5000 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [6]:
# Verify required inputs exist. The TTS tokenizer is committed under models/.
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")

All inputs ready.


## Reporting helpers

In [7]:
import json
from pathlib import Path


def load_result(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing result JSON: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def fmt(value, digits=4):
    if value is None:
        return "n/a"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def result_paths(model_slug):
    return {
        "random": Path("results") / f"run-{model_slug}.json",
        "mean_subword": Path("results") / f"run-{model_slug}-meansub.json",
    }


def print_tokenization_table(results):
    print("\nTOKENIZATION / FERTILITY")
    print("tokens/word is fertility; lower is better.")
    header = (
        f"{'arm':<22}{'base fertility':>16}{'Akan fertility':>16}"
        f"{'base tokens':>14}{'Akan tokens':>14}{'words':>10}{'reduction':>12}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        counts = payload["token_count_comparison"]
        base = counts["base_model_tokenizer"]
        akan = counts["experiment_tokenizer"]
        print(
            f"{arm:<22}"
            f"{fmt(base.get('fertility')):>16}"
            f"{fmt(akan.get('fertility')):>16}"
            f"{base.get('total_tokens'):>14}"
            f"{akan.get('total_tokens'):>14}"
            f"{akan.get('total_words'):>10}"
            f"{fmt(counts.get('token_reduction_ratio')):>12}"
        )


def print_bpb_table(results):
    print("\nMODELING / BPB")
    print("BPB uses full byte coverage; lower experiment BPB is better.")
    header = (
        f"{'arm':<22}{'eval_loss':>12}{'perplexity':>12}"
        f"{'experiment BPB':>18}{'base BPB':>12}{'improvement':>14}"
    )
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        bpb = payload["eval"]["bpb"]
        base_bpb = (bpb.get("base") or {}).get("bits_per_byte")
        print(
            f"{arm:<22}"
            f"{fmt(payload['eval'].get('eval_loss')):>12}"
            f"{fmt(payload['eval'].get('perplexity'), 2):>12}"
            f"{fmt(bpb['experiment'].get('bits_per_byte')):>18}"
            f"{fmt(base_bpb):>12}"
            f"{fmt(bpb.get('improvement')):>14}"
        )


def print_generation_samples(results):
    print("\nGENERATION SAMPLES")
    for arm, payload in results.items():
        print(f"\n[{arm}]")
        for index, sample in enumerate(payload.get("generation_samples", []), start=1):
            print(f"sample {index} prompt: {sample.get('prompt', '')}")
            print(f"sample {index} completion: {sample.get('completion', '')}")


def print_run_integrity(results):
    print("\nRUN INTEGRITY")
    header = f"{'arm':<22}{'output_model_dir':<34}{'reload ok':>10}{'device':>18}"
    print(header)
    print("-" * len(header))
    for arm, payload in results.items():
        reload_info = payload.get("reload_verification") or {}
        device = payload.get("device") or {}
        print(
            f"{arm:<22}"
            f"{str(payload.get('output_model_dir')):<34}"
            f"{str(reload_info.get('success')):>10}"
            f"{str(device.get('device_name')):>18}"
        )


def print_full_json(results):
    for arm, payload in results.items():
        print(f"\nBEGIN_FULL_JSON {arm}")
        print(json.dumps(payload, ensure_ascii=False, indent=2))
        print(f"END_FULL_JSON {arm}")


def display_model_results(model_slug):
    paths = result_paths(model_slug)
    results = {arm: load_result(path) for arm, path in paths.items()}
    print("=" * 88)
    print(f"FULL RESULTS FOR {model_slug}")
    print("=" * 88)
    print("Result files:")
    for arm, path in paths.items():
        print(f"- {arm}: {path}")
    print_tokenization_table(results)
    print_bpb_table(results)
    print_generation_samples(results)
    print_run_integrity(results)
    best = min(results, key=lambda arm: results[arm]["eval"]["bpb"]["experiment"]["bits_per_byte"])
    print(f"\nBest downstream BPB arm: {best}")
    print_full_json(results)
    return results


## Run qwen-1.7b

Model id: `Qwen/Qwen3-1.7B-Base`. This section runs both arms, then immediately displays and archives the full results.

In [8]:
# qwen-1.7b / Arm A - random embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-1.7B-Base

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.81MB/s]
tokenizer_config.json: 9.68kB [00:00, 18.3MB/s]
vocab.json: 2.78MB [00:00, 51.6MB/s]
merges.txt: 1.67MB [00:00, 150MB/s]
tokenizer.json: 7.03MB [00:00, 174MB/s]
Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1816.47text/s]
model.safetensors: 100%|████████████████████| 3.44G/3.44G [00:20<00:00, 167MB/s]
Loading weights: 100%|█| 310/310 [00:01<00:00, 238.85it/s, Materializing param=m
generation_config.json: 100%|███████████████████| 138/138 [00:00<00:00, 722kB/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg

In [9]:
# qwen-1.7b / Arm B - mean-of-subword embedding init
!python scripts/model_integration.py --model-id Qwen/Qwen3-1.7B-Base --embedding-init-mode mean_subword

Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1825.31text/s]
Loading weights: 100%|█| 310/310 [00:01<00:00, 252.52it/s, Materializing param=m
Initializing mean-subword embeddings: 100%|█| 8000/8000 [00:01<00:00, 4203.25tok
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '6.826', 'grad_norm': '5.481', 'learning_rate': '0.000193', 'epoch': '0.03906'}
{'loss': '5.658', 'grad_norm': '3.256', 'learning_rate': '0.0001852', 'epoch': '0.07812'}
{'loss': '5.333', 'grad_norm': '2.212', 'learning_rate': '0.0001773', 'epoch': '0.1172'}
{'loss': '5.039', 'grad_norm': '2.541', 'learning_rat

## Full results and artifact zip - qwen-1.7b

In [10]:
results_qwen_1_7b = display_model_results("qwen-1.7b")

FULL RESULTS FOR qwen-1.7b
Result files:
- random: results/run-qwen-1.7b.json
- mean_subword: results/run-qwen-1.7b-meansub.json

TOKENIZATION / FERTILITY
tokens/word is fertility; lower is better.
arm                     base fertility  Akan fertility   base tokens   Akan tokens     words   reduction
--------------------------------------------------------------------------------------------------------
random                          2.5303          1.2586        357830        177986    141420      0.5026
mean_subword                    2.5303          1.2586        357830        177986    141420      0.5026

MODELING / BPB
BPB uses full byte coverage; lower experiment BPB is better.
arm                      eval_loss  perplexity    experiment BPB    base BPB   improvement
------------------------------------------------------------------------------------------
random                      4.4110       82.36            1.4741      2.7556        1.2815
mean_subword                3.71

In [11]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-qwen-1.7b-artifacts.zip     results/run-qwen-1.7b.json     results/run-qwen-1.7b-meansub.json     models/run-qwen-1.7b     models/run-qwen-1.7b-meansub
!ls -lh outputs

models:
total 1.8M
-rw-r--r-- 1 root root 522K Jun 14 07:31 asr_tokenizer.json
-rw-r--r-- 1 root root 1.3K Jun 14 07:31 asr_tokenizer_stats.json
-rw-r--r-- 1 root root 515K Jun 14 07:31 mixed_tokenizer.json
-rw-r--r-- 1 root root  12K Jun 14 07:31 mixed_tokenizer_stats.json
-rw-r--r-- 1 root root 227K Jun 14 07:31 router_classifier.pkl
drwxr-xr-x 3 root root 4.0K Jun 14 08:30 run-qwen-1.7b
drwxr-xr-x 3 root root 4.0K Jun 14 09:24 run-qwen-1.7b-meansub
-rw-r--r-- 1 root root 515K Jun 14 07:31 tts_tokenizer.json
-rw-r--r-- 1 root root  11K Jun 14 07:31 tts_tokenizer_stats.json

outputs:
total 0

results:
total 8.0K
-rw-r--r-- 1 root root 3.4K Jun 14 08:30 run-qwen-1.7b.json
-rw-r--r-- 1 root root 3.4K Jun 14 09:25 run-qwen-1.7b-meansub.json
  adding: results/run-qwen-1.7b.json (deflated 66%)
  adding: results/run-qwen-1.7b-meansub.json (deflated 66%)
  adding: models/run-qwen-1.7b/ (stored 0%)
  adding: models/run-qwen-1.7b/adapter_model.safetensors (deflated 21%)
  adding: models/run-qw

## Run gemma-1b

Model id: `google/gemma-3-1b-pt`. This section runs both arms, then immediately displays and archives the full results.

In [12]:
# gemma-1b / Arm A - random embedding init
!python scripts/model_integration.py --model-id google/gemma-3-1b-pt

config.json: 100%|█████████████████████████████| 880/880 [00:00<00:00, 4.20MB/s]
tokenizer_config.json: 1.16MB [00:00, 61.4MB/s]
tokenizer.json: 100%|██████████████████████| 33.4M/33.4M [00:01<00:00, 27.2MB/s]
added_tokens.json: 100%|██████████████████████| 35.0/35.0 [00:00<00:00, 204kB/s]
Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1759.64text/s]
model.safetensors: 100%|████████████████████| 2.00G/2.00G [00:15<00:00, 128MB/s]
Loading weights: 100%|█| 340/340 [00:00<00:00, 394.03it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 215/215 [00:00<00:00, 1.30MB/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com

In [13]:
# gemma-1b / Arm B - mean-of-subword embedding init
!python scripts/model_integration.py --model-id google/gemma-3-1b-pt --embedding-init-mode mean_subword

Counting Akan tokens: 100%|███████████████| 512/512 [00:00<00:00, 1578.11text/s]
Loading weights: 100%|█| 340/340 [00:00<00:00, 423.74it/s, Materializing param=m
Initializing mean-subword embeddings: 100%|█| 8000/8000 [00:01<00:00, 4428.31tok
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '7.021', 'grad_norm': '4.546', 'learning_rate': '0.000193', 'epoch': '0.03906'}
{'loss': '5.635', 'grad_norm': '2.113', 'learning_rate': '0.0001852', 'epoch': '0.07812'}
{'loss': '5.312', 'grad_norm': '1.936', 'learning_rate': '0.0001773', 'epoch': '0.1172'}
{'loss': '5.038', 'grad_norm': '2.211', 'learning_rat

## Full results and artifact zip - gemma-1b

In [14]:
results_gemma_1b = display_model_results("gemma-1b")

FULL RESULTS FOR gemma-1b
Result files:
- random: results/run-gemma-1b.json
- mean_subword: results/run-gemma-1b-meansub.json

TOKENIZATION / FERTILITY
tokens/word is fertility; lower is better.
arm                     base fertility  Akan fertility   base tokens   Akan tokens     words   reduction
--------------------------------------------------------------------------------------------------------
random                          2.2841          1.2586        323024        177986    141420      0.4490
mean_subword                    2.2841          1.2586        323024        177986    141420      0.4490

MODELING / BPB
BPB uses full byte coverage; lower experiment BPB is better.
arm                      eval_loss  perplexity    experiment BPB    base BPB   improvement
------------------------------------------------------------------------------------------
random                      4.4675       87.14            1.4841      3.3907        1.9067
mean_subword                3.6706 

In [15]:
!mkdir -p outputs
!ls -lh results models outputs
!zip -r outputs/run-gemma-1b-artifacts.zip     results/run-gemma-1b.json     results/run-gemma-1b-meansub.json     models/run-gemma-1b     models/run-gemma-1b-meansub
!ls -lh outputs

models:
total 1.8M
-rw-r--r-- 1 root root 522K Jun 14 07:31 asr_tokenizer.json
-rw-r--r-- 1 root root 1.3K Jun 14 07:31 asr_tokenizer_stats.json
-rw-r--r-- 1 root root 515K Jun 14 07:31 mixed_tokenizer.json
-rw-r--r-- 1 root root  12K Jun 14 07:31 mixed_tokenizer_stats.json
-rw-r--r-- 1 root root 227K Jun 14 07:31 router_classifier.pkl
drwxr-xr-x 3 root root 4.0K Jun 14 10:21 run-gemma-1b
drwxr-xr-x 3 root root 4.0K Jun 14 11:15 run-gemma-1b-meansub
drwxr-xr-x 3 root root 4.0K Jun 14 08:30 run-qwen-1.7b
drwxr-xr-x 3 root root 4.0K Jun 14 09:24 run-qwen-1.7b-meansub
-rw-r--r-- 1 root root 515K Jun 14 07:31 tts_tokenizer.json
-rw-r--r-- 1 root root  11K Jun 14 07:31 tts_tokenizer_stats.json

outputs:
total 2.1G
-rw-r--r-- 1 root root 2.1G Jun 14 09:28 run-qwen-1.7b-artifacts.zip

results:
total 16K
-rw-r--r-- 1 root root 3.5K Jun 14 10:21 run-gemma-1b.json
-rw-r--r-- 1 root root 3.4K Jun 14 11:16 run-gemma-1b-meansub.json
-rw-r--r-- 1 root root 3.4K Jun 14 08:30 run-qwen-1.7b.json
-rw-r-

## Notebook-level summary

In [16]:
MODEL_SLUGS = ['qwen-1.7b', 'gemma-1b']
all_results = {slug: {arm: load_result(path) for arm, path in result_paths(slug).items()} for slug in MODEL_SLUGS}

print("=" * 88)
print("NOTEBOOK SPLIT SUMMARY")
print("=" * 88)
print_tokenization_table({f"{slug}/{arm}": payload for slug, arms in all_results.items() for arm, payload in arms.items()})
print_bpb_table({f"{slug}/{arm}": payload for slug, arms in all_results.items() for arm, payload in arms.items()})

summary = {
    slug: {
        arm: {
            "eval_loss": payload["eval"]["eval_loss"],
            "perplexity": payload["eval"]["perplexity"],
            "experiment_bpb": payload["eval"]["bpb"]["experiment"]["bits_per_byte"],
            "base_bpb": (payload["eval"]["bpb"].get("base") or {}).get("bits_per_byte"),
            "bpb_improvement": payload["eval"]["bpb"].get("improvement"),
            "base_fertility": payload["token_count_comparison"]["base_model_tokenizer"]["fertility"],
            "akan_fertility": payload["token_count_comparison"]["experiment_tokenizer"]["fertility"],
            "token_reduction_ratio": payload["token_count_comparison"]["token_reduction_ratio"],
        }
        for arm, payload in arms.items()
    }
    for slug, arms in all_results.items()
}
Path("outputs").mkdir(exist_ok=True)
Path("outputs/heavy-split-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote outputs/heavy-split-summary.json")


NOTEBOOK SPLIT SUMMARY

TOKENIZATION / FERTILITY
tokens/word is fertility; lower is better.
arm                     base fertility  Akan fertility   base tokens   Akan tokens     words   reduction
--------------------------------------------------------------------------------------------------------
qwen-1.7b/random                2.5303          1.2586        357830        177986    141420      0.5026
qwen-1.7b/mean_subword          2.5303          1.2586        357830        177986    141420      0.5026
gemma-1b/random                 2.2841          1.2586        323024        177986    141420      0.4490
gemma-1b/mean_subword           2.2841          1.2586        323024        177986    141420      0.4490

MODELING / BPB
BPB uses full byte coverage; lower experiment BPB is better.
arm                      eval_loss  perplexity    experiment BPB    base BPB   improvement
------------------------------------------------------------------------------------------
qwen-1.7b/random   

In [17]:
!ls -lh outputs

total 3.3G
-rw-r--r-- 1 root root 1.5K Jun 14 11:18 heavy-split-summary.json
-rw-r--r-- 1 root root 1.3G Jun 14 11:18 run-gemma-1b-artifacts.zip
-rw-r--r-- 1 root root 2.1G Jun 14 09:28 run-qwen-1.7b-artifacts.zip
